# Imports

In [ ]:
import scanpy as sc
import scanpy.external as sce
import numpy as np
import pandas as pd
import warnings, scipy.sparse as sp, matplotlib, matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.pyplot import rc_context
from collections import Counter
import matplotlib.font_manager
import re
import matplotlib.lines as lines

pd.set_option('display.max_rows', 200)

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = 'Arial'
matplotlib.rc('font', size=14)

sc.settings.n_jobs=-1
sc.set_figure_params(dpi=100, dpi_save=300, color_map='Spectral_r', vector_friendly=True, transparent=True)
sc.settings.figdir = "../2_figures/"
sc.settings.verbosity = 1 # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()

In [ ]:
def observe_variance(anndata_object):
    fig = plt.figure(figsize=(10,5))
    ax1 = fig.add_subplot(121)
    ax2 = fig.add_subplot(122)
    # variance per principal component
    x = range(len(anndata_object.uns['pca']['variance_ratio']))
    y = anndata_object.uns['pca']['variance_ratio']
    ax1.scatter(x,y,s=4)
    ax1.set_xlabel('PC')
    ax1.set_ylabel('Fraction of variance explained\n')
    ax1.set_title('Fraction of variance explained per PC\n')
    # cumulative variance explained
    cml_var_explained = np.cumsum(anndata_object.uns['pca']['variance_ratio'])
    x = range(len(anndata_object.uns['pca']['variance_ratio']))
    y = cml_var_explained
    ax2.scatter(x,y,s=4)
    ax2.set_xlabel('PC')
    ax2.set_ylabel('Cumulative fraction of variance\nexplained')
    ax2.set_title('Cumulative fraction of variance\nexplained by PCs')
    fig.tight_layout()
    plot = plt.show
    return(plot)

In [ ]:
%matplotlib inline 

In [ ]:
# preset color palettes and color maps
user_defined_palette =  [ '#F6222E', '#16FF32', '#3283FE', '#FEAF16', '#BDCDFF', '#3B00FB', '#1CFFCE', '#C075A6', '#F8A19F', '#B5EFB5', '#FBE426', '#C4451C', 
                          '#2ED9FF', '#c1c119', '#8b0000', '#FE00FA', '#1CBE4F', '#1C8356', '#0e452b', '#AA0DFE', '#B5EFB5', '#325A9B', '#90AD1C']

user_defined_cmap_markers = LinearSegmentedColormap.from_list('mycmap', ["#E6E6FF", "#CCCCFF", "#B2B2FF", "#9999FF",  "#6666FF",   "#3333FF", "#0000FF"])
user_defined_cmap_degs = LinearSegmentedColormap.from_list('mycmap', ["#0000FF", "#3333FF", "#6666FF", "#9999FF", "#B2B2FF", "#CCCCFF", "#E6E6FF", "#E6FFE6", "#CCFFCC", "#B2FFB2", "#99FF99", "#66FF66", "#33FF33", "#00FF00"])

In [ ]:
file_outputs = '../../1_outputs/2_deg/'
h5ad = '../../1_outputs/1_h5ad/'

In [ ]:
#adata = sc.read_h5ad('../3_h5ad/4_dn_subset.h5ad')
adata = sc.read_h5ad('../../1_outputs/1_h5ad/4_dn_dp.h5ad')

# Annotations 

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=1500, n_bins=20, flavor='seurat',  inplace=True)

In [ ]:
sc.tl.pca(adata, n_comps=50, svd_solver='arpack', random_state=42, use_highly_variable=True)

In [ ]:
observe_variance(adata)

In [ ]:
sc.tl.pca(adata, n_comps=75, svd_solver='arpack', random_state=42, use_highly_variable=True)

In [ ]:
sce.pp.harmony_integrate(adata, ['sample'], max_iter_harmony = 20)

In [ ]:
sc.pp.neighbors(adata, n_neighbors=30, random_state=42, use_rep='X_pca_harmony')

In [ ]:
sc.tl.umap(adata, random_state=42) #min_dist=0.2, spread = 1.3,

In [ ]:
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    adata, 
    color=['condition', 'sample',],   
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False
)

In [ ]:
''' 
https://www.frontiersin.org/journals/immunology/articles/10.3389/fimmu.2022.884569/full
https://onlinelibrary.wiley.com/doi/10.1155/2012/467101
https://www.nature.com/articles/nri913
https://pmc.ncbi.nlm.nih.gov/articles/PMC3131407/
https://www.biorxiv.org/content/10.1101/2022.02.25.481936v4.full
https://www.sciencedirect.com/science/article/pii/S0301472X20306949
'''
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    adata, 
    color=['Flt3', 'Hoxa9', 'Mef2c', 'Kit', 'Cd44','Bcl11a', 'Lyl1','Il7r',#DN1
           'Il2ra', 'Bcl11b', 'Cd3g', 'Cd3d', 'Cd3e', 'Dtx1', 'Gata3', #DN2
           'Lef1', 'Hes1', 'Notch1', 'Runx1', 'Myb', 'Ets1', 'Ets2', 'Id3', 'Cd28', 'Ptcra', 'Notch3', 'Ccr9', 'Gzma', 'Tal1', 'Hes1', #DN3,
           'Rag1', 'Rag2', 'Bcl2', 'Bcl6', 'Ikzf1', 'Gata2', #DN4
           'Cd4', 'Cd8a', 'Nfatc1', #DP
           'Sox13','Rorc', 'Trdc', 'Il2rb' , #Gamma Delta 
           'Mki67',
           'Tcf12', # Myeloid Precursors
          ],   
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False, 
    vmax='p99'
)

In [ ]:
''' 
https://www.frontiersin.org/journals/immunology/articles/10.3389/fimmu.2022.884569/full
https://onlinelibrary.wiley.com/doi/10.1155/2012/467101
https://www.nature.com/articles/nri913
https://pmc.ncbi.nlm.nih.gov/articles/PMC3131407/
https://www.biorxiv.org/content/10.1101/2022.02.25.481936v4.full
https://www.sciencedirect.com/science/article/pii/S0301472X20306949
'''
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    adata, 
    color=[

        'condition',
        
        'Hoxa9', 'Kit', 'Cd44',
        
        'Il2ra', 'Cd3d', 'Rag1',
        
        'Cd4', 'Cd8a', 'Cd8b1', 'Mki67', 

        'Trdc', 'Il2ra', 'Cd24a', 'Trgv2',
        
         
        
          ],   
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False, 
    vmax='p99'
)

In [ ]:
''' 
https://www.frontiersin.org/journals/immunology/articles/10.3389/fimmu.2022.884569/full
https://onlinelibrary.wiley.com/doi/10.1155/2012/467101
https://www.nature.com/articles/nri913
https://pmc.ncbi.nlm.nih.gov/articles/PMC3131407/
https://www.biorxiv.org/content/10.1101/2022.02.25.481936v4.full
https://www.sciencedirect.com/science/article/pii/S0301472X20306949
https://www.researchgate.net/figure/Single-cell-RNA-sequencing-scRNA-seq-of-cells-encompassing-all-stages-of-gd-T-cell_fig1_329230935
'''
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    adata, 
    color=[
           'Il2ra', 'Bcl11b',  'Bcl11a', 'Cd3g', 'Cd3d', 'Cd3e', 'Dtx1', 'Gata3', 'Spi1', #DN2
           'Notch1', 'Ly6d', 'Lat', 'Lck', 'Cd27', 'Cd5', 'Hdac4', 'Tcf7', 'Mpo', #DN3
           'Rag1', 'Rag2', 'Bcl2', 'Bcl6', 'Ikzf1', 'Gata2', #DN4
           'Cd4', 'Cd8a', 'Nfatc1', #DP
           'Mki67', 'Runx1', 'Ets1'
          ],   
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False, 
    vmax='p99'
)

In [ ]:
for resolution_parameter in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 
                             1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0]: #0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0
    sc.tl.leiden(adata, resolution=resolution_parameter, random_state=42, 
                        key_added='leiden_'+str(resolution_parameter))

In [ ]:
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4',
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    adata, 
    color=['leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4', 'leiden_0.5',
           'leiden_0.6', 'leiden_0.7', 'leiden_0.8','leiden_0.9', 'leiden_1.0',
           'leiden_1.1', 'leiden_1.2', 
           'leiden_1.3','leiden_1.4', 'leiden_1.5', 
        'leiden_1.6', 
           'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'
          ], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False
)

In [ ]:
#adata.write_h5ad(h5ad + '10_dn_dp.h5ad')

In [ ]:
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4',
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    adata, 
    color=['leiden_0.1',
          ], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False,
    legend_loc='on data'
)

In [ ]:
cell_type_groups = {
    'sub1': ['1', '2', '3', '4'],
    'sub2': ['0',], 
}

cluster_to_cell_type = {cluster: cell_type for cell_type, clusters in cell_type_groups.items() for cluster in clusters}

In [ ]:
adata.obs['cell_type_subset'] = adata.obs['leiden_0.1'].map(cluster_to_cell_type)

In [ ]:
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    adata, 
    color=['cell_type_subset', 
          ],
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False, 
)

In [ ]:
#adata.write_h5ad(h5ad + '10_dn_dp.h5ad.h5ad')

## Sub1

In [ ]:
subset1 = adata[adata.obs['cell_type_subset'].isin(['sub1'])].copy()
subset1

In [ ]:
sc.pp.highly_variable_genes(subset1, n_top_genes=1000, n_bins=20, flavor='seurat',  inplace=True)

In [ ]:
sc.tl.pca(subset1, n_comps=50, svd_solver='arpack', random_state=42, use_highly_variable=True)

In [ ]:
observe_variance(subset1)

In [ ]:
sc.tl.pca(subset1, n_comps=75, svd_solver='arpack', random_state=42, use_highly_variable=True)

In [ ]:
sce.pp.harmony_integrate(subset1, ['sample'], max_iter_harmony = 20)

In [ ]:
sc.pp.neighbors(subset1, n_neighbors=30, random_state=42, use_rep='X_pca_harmony')

In [ ]:
sc.tl.umap(subset1, random_state=42)#, min_dist=0.2, spread = 1.3, ) #min_dist=0.2, spread = 1.3, 

In [ ]:
''' 
https://www.frontiersin.org/journals/immunology/articles/10.3389/fimmu.2022.884569/full
https://onlinelibrary.wiley.com/doi/10.1155/2012/467101
https://www.nature.com/articles/nri913
https://pmc.ncbi.nlm.nih.gov/articles/PMC3131407/
https://www.biorxiv.org/content/10.1101/2022.02.25.481936v4.full
https://www.sciencedirect.com/science/article/pii/S0301472X20306949
'''
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    subset1, 
    color=['condition',
           'Hoxa9', 'Mef2c', 'Kit', 'Cd44','Bcl11a',#DN1
           'Il7r', 'Il2ra', 'Bcl11b', 'Gata3', 'Cd3e', 'Bcl2', #DN2
           'Ets2', 'Ccr9', 'Rag1', 'Hes1', 'Notch1', #DN3
           'Myc', 'Mki67', 'Lef1', 'Gzma', 'Ikzf1', #DN4
           'Cd4', 'Cd8a', 'Top2a', 'Hells', #DP(P)

            'Notch1', 'Ly6d', 'Lat', 'Lck', 'Cd27', 'Cd5', 'Hdac4', 'Tcf7', 'Mpo', #DN3

            'Trdc', 'Trgv2', 'Foxp3'
          ],   
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False, 
    vmax='p99'
)

In [ ]:
#0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 
#1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0

for resolution_parameter in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5]:
    sc.tl.leiden(subset1, resolution=resolution_parameter, random_state=42, key_added='leiden_'+str(resolution_parameter))

In [ ]:
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4', 'leiden_0.5', 'leiden_0.6', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'

sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    subset1, 
    color=[
        'leiden_0.1', 'leiden_0.2', 'leiden_0.3', 'leiden_0.4', 'leiden_0.5',
        'leiden_0.6', 'leiden_0.7', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
        'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5',
           ], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=4,
    wspace = 0.7,
    outline_width=[0.6, 0.05],
    #size=35,
    frameon=False,
    add_outline=True,
    sort_order = False
)

In [ ]:
sc.tl.rank_genes_groups(subset1, 
                        groupby='leiden_0.9',  # Change the Leiden clustering
                        method='wilcoxon', 
                        use_raw=False)

result = subset1.uns['rank_genes_groups']
groups = result['names'].dtype.names

df = pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names']}).head(150)

#df.to_csv(file_outputs + 'dn_dp_subset1.csv', index = False)

df.head(35)

In [ ]:
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4',
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    subset1, 
    color=['leiden_0.2'], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False,
    legend_loc='on data'
)

In [ ]:
cell_type_groups = {
    'other': ['3'],
    'DN_DP': ['0', '1', '2', '4']
}

cluster_to_cell_type = {cluster: cell_type for cell_type, clusters in cell_type_groups.items() for cluster in clusters}

In [ ]:
# cell_type_groups = {
#     'T:DN1': ['14'],
#     'T:DN2': ['8', '4'],
#     'T:DN3': ['2', '13'],
#     'T:DN4': ['0', '1', '3', '5', '6', '19', '9', '11', '12', '18'],
#     'T:DP(P)': ['7', '10', '15', '16', '17']
# }

# cluster_to_cell_type = {cluster: cell_type for cell_type, clusters in cell_type_groups.items() for cluster in clusters}

In [ ]:
subset1.obs['cell_type_subset'] = subset1.obs['leiden_0.2'].map(cluster_to_cell_type).fillna('Other')

In [ ]:
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    subset1, 
    color=['cell_type_subset', 
          ],
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False, 
)

### Other Subset

In [ ]:
other = subset1[subset1.obs['cell_type_subset'].isin(['other'])].copy()
other

In [ ]:
sc.pp.highly_variable_genes(other, n_top_genes=1000, n_bins=20, flavor='seurat',  inplace=True)

In [ ]:
sc.tl.pca(other, n_comps=50, svd_solver='arpack', random_state=42, use_highly_variable=True)

In [ ]:
observe_variance(other)

In [ ]:
sc.tl.pca(other, n_comps=75, svd_solver='arpack', random_state=42, use_highly_variable=True)

In [ ]:
sce.pp.harmony_integrate(other, ['sample'], max_iter_harmony = 20)

In [ ]:
sc.pp.neighbors(other, n_neighbors=30, random_state=42, use_rep='X_pca_harmony')

In [ ]:
sc.tl.umap(other, random_state=42)#, min_dist=0.2, spread = 1.3, ) #min_dist=0.2, spread = 1.3, 

In [ ]:
''' 
https://www.frontiersin.org/journals/immunology/articles/10.3389/fimmu.2022.884569/full
https://onlinelibrary.wiley.com/doi/10.1155/2012/467101
https://www.nature.com/articles/nri913
https://pmc.ncbi.nlm.nih.gov/articles/PMC3131407/
https://www.biorxiv.org/content/10.1101/2022.02.25.481936v4.full
https://www.sciencedirect.com/science/article/pii/S0301472X20306949
'''
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    other, 
    color=['condition',
           'Hoxa9', 'Mef2c', 'Kit', 'Cd44','Bcl11a',#DN1
           'Il7r', 'Il2ra', 'Bcl11b', 'Gata3', 'Cd3e', 'Bcl2', #DN2
           'Ets2', 'Ccr9', 'Rag1', 'Hes1', 'Notch1', #DN3
           'Myc', 'Mki67', 'Lef1', 'Gzma', 'Ikzf1', #DN4
           'Cd4', 'Cd8a', 'Top2a', 'Hells', #DP(P)

            'Notch1', 'Ly6d', 'Lat', 'Lck', 'Cd27', 'Cd5', 'Hdac4', 'Tcf7', 'Mpo', #DN3

            'Trdc', 'Trgv2', 'Foxp3', 'Mki67', 'Trac'
          ],   
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False, 
    vmax='p99'
)

In [ ]:
#0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 
#1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0

for resolution_parameter in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 
                             1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0]:
    sc.tl.leiden(other, resolution=resolution_parameter, random_state=42, key_added='leiden_'+str(resolution_parameter))

In [ ]:
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4', 'leiden_0.5', 'leiden_0.6', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'

sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    other, 
    color=[
        'leiden_0.1', 'leiden_0.2', 'leiden_0.3', 'leiden_0.4', 'leiden_0.5',
        'leiden_0.6', 'leiden_0.7', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
        'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5',
        'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'
           ], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=4,
    wspace = 0.7,
    outline_width=[0.6, 0.05],
    #size=35,
    frameon=False,
    add_outline=True,
    sort_order = False
)

In [ ]:
sc.tl.rank_genes_groups(other, 
                        groupby='leiden_0.9',  # Change the Leiden clustering
                        method='wilcoxon', 
                        use_raw=False)

result = other.uns['rank_genes_groups']
groups = result['names'].dtype.names

df = pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names']}).head(150)

#df.to_csv(file_outputs + 'dn_dp_subset1.csv', index = False)

df.head(35)

In [ ]:
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4',
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    other, 
    color=['leiden_0.9', 'Foxp3','Cd4', 'Cd8a'], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False,
    legend_loc='on data'
)

In [ ]:
cell_type_groups = {
    'T:Agonist': ['0'], 
    'T:CD4': ['3', '8'], 
    'T:Treg': ['9'], 
    'T:GD': ['1', '2', '4', '5', '6', '7'], 
    
}

cluster_to_cell_type = {cluster: cell_type for cell_type, clusters in cell_type_groups.items() for cluster in clusters}

In [ ]:
other.obs['cell_type_subset'] = other.obs['leiden_0.9'].map(cluster_to_cell_type).fillna('Other')

In [ ]:
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    other, 
    color=['cell_type_subset', 
          ],
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False, 
)

### DN_DP Subset

In [ ]:
dn_dp_sub = subset1[subset1.obs['cell_type_subset'].isin(['DN_DP'])].copy()
dn_dp_sub

In [ ]:
sc.pp.highly_variable_genes(dn_dp_sub, n_top_genes=1000, n_bins=20, flavor='seurat',  inplace=True)

In [ ]:
sc.tl.pca(dn_dp_sub, n_comps=50, svd_solver='arpack', random_state=42, use_highly_variable=True)

In [ ]:
observe_variance(dn_dp_sub)

In [ ]:
sc.tl.pca(dn_dp_sub, n_comps=75, svd_solver='arpack', random_state=42, use_highly_variable=True)

In [ ]:
sce.pp.harmony_integrate(dn_dp_sub, ['sample',], max_iter_harmony = 20)

In [ ]:
sc.pp.neighbors(dn_dp_sub, n_neighbors=30, random_state=42, use_rep='X_pca_harmony')

In [ ]:
sc.tl.umap(dn_dp_sub, random_state=42)#, min_dist=0.2, spread = 1.3, ) #min_dist=0.2, spread = 1.3, 

In [ ]:
''' 
https://www.frontiersin.org/journals/immunology/articles/10.3389/fimmu.2022.884569/full
https://onlinelibrary.wiley.com/doi/10.1155/2012/467101
https://www.nature.com/articles/nri913
https://pmc.ncbi.nlm.nih.gov/articles/PMC3131407/
https://www.biorxiv.org/content/10.1101/2022.02.25.481936v4.full
https://www.sciencedirect.com/science/article/pii/S0301472X20306949
'''
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    dn_dp_sub, 
    color=['condition',
           'Hoxa9', 'Mef2c', 'Kit', 'Cd44','Bcl11a',#DN1
           'Il7r', 'Il2ra', 'Bcl11b', 'Gata3', 'Cd3e', 'Bcl2', #DN2
           'Ets2', 'Ccr9', 'Rag1', 'Hes1', 'Notch1', #DN3
           'Myc', 'Mki67', 'Lef1', 'Gzma', 'Ikzf1', #DN4
           'Cd4', 'Cd8a', 'Top2a', 'Hells', #DP(P)

            'Notch1', 'Ly6d', 'Lat', 'Lck', 'Cd27', 'Cd5', 'Hdac4', 'Tcf7', 'Mpo', #DN3

            'Trdc', 'Trgv2', 'Foxp3', 'Mki67', 'Trac'
          ],   
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False, 
    vmax='p99'
)

In [ ]:
#0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 
#1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0

for resolution_parameter in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 
                             1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0,
                             2.1, 2.2, 2.3, 2.4, 2.5, 2.6, 2.7, 2.8, 2.9, 3.0
                             ]:
    sc.tl.leiden(dn_dp_sub, resolution=resolution_parameter, random_state=42, key_added='leiden_'+str(resolution_parameter))

In [ ]:
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4', 'leiden_0.5', 'leiden_0.6', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'

sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    dn_dp_sub, 
    color=[
        'leiden_0.1', 'leiden_0.2', 'leiden_0.3', 'leiden_0.4', 'leiden_0.5',
        'leiden_0.6', 'leiden_0.7', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
        'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5',
        'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0',
        #'leiden_2.1', 'leiden_2.2', 'leiden_2.3','leiden_2.4', 'leiden_2.5',
        #'leiden_2.6', 'leiden_2.7', 'leiden_2.8', 'leiden_2.9', 'leiden_3.0'
           ], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=4,
    wspace = 0.7,
    outline_width=[0.6, 0.05],
    #size=35,
    frameon=False,
    add_outline=True,
    sort_order = False
)

In [ ]:
sc.tl.rank_genes_groups(dn_dp_sub, 
                        groupby='leiden_0.8',  # Change the Leiden clustering
                        method='wilcoxon', 
                        use_raw=False)

result = other.uns['rank_genes_groups']
groups = result['names'].dtype.names

df = pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names']}).head(150)

#df.to_csv(file_outputs + 'dn_dp_subset1.csv', index = False)

df.head(35)

In [ ]:
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4',
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    dn_dp_sub, 
    color=['leiden_1.3'], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False,
    legend_loc='on data'
)

In [ ]:
cell_type_groups = {
    'T:DN1': ['12'],
    'T:DN2': ['4', '7', '2'], 
    'T:DN3': ['1', '6', '0'], 
    'T:DN4': ['5', '3', '13', '9'],
    'T:DP(P)': ['8', '11'], 
    'SP': ['10']
    }

cluster_to_cell_type = {cluster: cell_type for cell_type, clusters in cell_type_groups.items() for cluster in clusters}

In [ ]:
dn_dp_sub.obs['cell_type_subset'] = dn_dp_sub.obs['leiden_1.3'].map(cluster_to_cell_type).fillna('Other')

In [ ]:
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    dn_dp_sub, 
    color=['cell_type_subset', 
          ],
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False, 
)

### Annotate SPs

In [ ]:
sp = dn_dp_sub[dn_dp_sub.obs['cell_type_subset'].isin(['SP'])].copy()
sp

In [ ]:
sc.pp.highly_variable_genes(sp, n_top_genes=1000, n_bins=20, flavor='seurat',  inplace=True)

In [ ]:
sc.tl.pca(sp, n_comps=50, svd_solver='arpack', random_state=42, use_highly_variable=True)

In [ ]:
observe_variance(sp)

In [ ]:
sc.tl.pca(sp, n_comps=75, svd_solver='arpack', random_state=42, use_highly_variable=True)

In [ ]:
sce.pp.harmony_integrate(sp, ['sample'], max_iter_harmony = 20)

In [ ]:
sc.pp.neighbors(sp, n_neighbors=30, random_state=42, use_rep='X_pca_harmony')

In [ ]:
sc.tl.umap(sp, random_state=42)#, min_dist=0.2, spread = 1.3, ) #min_dist=0.2, spread = 1.3, 

In [ ]:
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    sp, 
    color=['condition',
           'Cd4', 'Cd8a', 'Sell', 'Mki67'
          ],   
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False, 
    vmax='p99'
)

In [ ]:
#0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 
#1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0

for resolution_parameter in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 
                             1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0,
                             2.1, 2.2, 2.3, 2.4, 2.5, 2.6, 2.7, 2.8, 2.9, 3.0
                             ]:
    sc.tl.leiden(sp, resolution=resolution_parameter, random_state=42, key_added='leiden_'+str(resolution_parameter))

In [ ]:
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4', 'leiden_0.5', 'leiden_0.6', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'

sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    sp, 
    color=[
        'leiden_0.1', 'leiden_0.2', 'leiden_0.3', 'leiden_0.4', 'leiden_0.5',
        'leiden_0.6', 'leiden_0.7', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
        'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5',
        'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0',
        #'leiden_2.1', 'leiden_2.2', 'leiden_2.3','leiden_2.4', 'leiden_2.5',
        #'leiden_2.6', 'leiden_2.7', 'leiden_2.8', 'leiden_2.9', 'leiden_3.0'
           ], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=4,
    wspace = 0.7,
    outline_width=[0.6, 0.05],
    #size=35,
    frameon=False,
    add_outline=True,
    sort_order = False
)

In [ ]:
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4',
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    sp, 
    color=['leiden_0.7'], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False,
    legend_loc='on data'
)

In [ ]:
cell_type_groups = {
    'T:CD4': ['0', '2'], 
    'T:CD8': ['1', '3', '4'],
}

cluster_to_cell_type = {cluster: cell_type for cell_type, clusters in cell_type_groups.items() for cluster in clusters}

In [ ]:
sp.obs['cell_type_subset'] = sp.obs['leiden_0.7'].map(cluster_to_cell_type).fillna('Other')

In [ ]:
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    sp, 
    color=['cell_type_subset', 
          ],
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False, 
)

#### Map SP back to dn_dp_sub

In [ ]:
dn_dp_sub.obs['cell_type_subset'].unique()

### Fill in the brackets below with the subsets you annotated beyond just the broad annotation. 

In [ ]:
temp = dn_dp_sub[~dn_dp_sub.obs['cell_type_subset'].isin(['SP'])].obs['cell_type_subset'].copy()
temp

In [ ]:
with_renamed_subsets = pd.concat([sp.obs['cell_type_subset'],
                                  temp])

In [ ]:
dn_dp_sub.obs['cell_type_subset'] = ''

In [ ]:
dn_dp_sub.obs['cell_type_subset'][dn_dp_sub.obs.index.isin(with_renamed_subsets.index) == True] = with_renamed_subsets

In [ ]:
sc.pp.highly_variable_genes(dn_dp_sub, n_top_genes=1000, n_bins=20, flavor='seurat',  inplace=True)

In [ ]:
sc.tl.pca(dn_dp_sub, n_comps=50, svd_solver='arpack', random_state=42, use_highly_variable=False) #change use_highly_variable == True if you ran the block above

In [ ]:
observe_variance(dn_dp_sub)

In [ ]:
sc.tl.pca(dn_dp_sub, n_comps=75, svd_solver='arpack', random_state=42, use_highly_variable=True) #change use_highly_variable == True if you ran the block above

#### Harmony integrate the data if there are large batch effects

In [ ]:
sce.pp.harmony_integrate(dn_dp_sub, ['sample']) # Usually you want to run based on replicates but you can also run based on other parameters

If you batch corrected, make sure to use the use_rep = 'X_pca_harmony' paremeter below

In [ ]:
rng = np.random.RandomState(42)
sc.pp.neighbors(dn_dp_sub, n_neighbors=30, random_state=42, use_rep='X_pca_harmony')  # Change use_rep == X_pca_harmony if you ran the block above

In [ ]:
sc.tl.umap(dn_dp_sub, random_state=42)

In [ ]:
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    dn_dp_sub, 
    color=['sample', 'cell_type', 'cell_type_subset'],   
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.5,
    outline_width=[0.6, 0.05],
    #size=5,
    frameon=False,
    add_outline=True,
    sort_order = False
)

### Map Subsets Back to Sub1

In [ ]:
subset1.obs['cell_type_subset'].unique()

### Fill in the brackets below with the subsets you annotated beyond just the broad annotation. 

In [ ]:
temp = subset1[~subset1.obs['cell_type_subset'].isin(['DN_DP', 'other'])].obs['cell_type'].copy()
temp

In [ ]:
with_renamed_subsets = pd.concat([other.obs['cell_type_subset'],
                                  dn_dp_sub.obs['cell_type_subset'],
                                  temp])

In [ ]:
subset1.obs['cell_type_subset'] = ''

In [ ]:
subset1.obs['cell_type_subset'][subset1.obs.index.isin(with_renamed_subsets.index) == True] = with_renamed_subsets

In [ ]:
sc.pp.highly_variable_genes(subset1, n_top_genes=1000, n_bins=20, flavor='seurat',  inplace=True)

In [ ]:
sc.tl.pca(subset1, n_comps=100, svd_solver='arpack', random_state=42, use_highly_variable=True) #change use_highly_variable == True if you ran the block above

In [ ]:
observe_variance(subset1)

In [ ]:
sc.tl.pca(subset1, n_comps=75, svd_solver='arpack', random_state=42, use_highly_variable=True) #change use_highly_variable == True if you ran the block above

#### Harmony integrate the data if there are large batch effects

In [ ]:
sce.pp.harmony_integrate(subset1, ['sample']) # Usually you want to run based on replicates but you can also run based on other parameters

If you batch corrected, make sure to use the use_rep = 'X_pca_harmony' paremeter below

In [ ]:
rng = np.random.RandomState(42)
sc.pp.neighbors(subset1, n_neighbors=30, random_state=42, use_rep='X_pca_harmony')  # Change use_rep == X_pca_harmony if you ran the block above

In [ ]:
sc.tl.umap(subset1, random_state=42)

In [ ]:
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    subset1, 
    color=['sample', 'cell_type', 'cell_type_subset'],   
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.5,
    outline_width=[0.6, 0.05],
    size=5,
    frameon=False,
    add_outline=True,
    sort_order = False
)

## Subset 2

In [ ]:
subset2 = adata[adata.obs['cell_type_subset'].isin(['sub2'])].copy()
subset2

In [ ]:
sc.pp.highly_variable_genes(subset2, n_top_genes=1000, n_bins=20, flavor='seurat',  inplace=True)

In [ ]:
sc.tl.pca(subset2, n_comps=50, svd_solver='arpack', random_state=42, use_highly_variable=True)

In [ ]:
observe_variance(subset2)

In [ ]:
sc.tl.pca(subset2, n_comps=13, svd_solver='arpack', random_state=42, use_highly_variable=True)

In [ ]:
sce.pp.harmony_integrate(subset2, 'sample', max_iter_harmony = 20)

In [ ]:
sc.pp.neighbors(subset2, n_neighbors=25, random_state=42, use_rep='X_pca_harmony')

In [ ]:
sc.tl.umap(subset2, random_state=42)#, min_dist=0.2, spread = 1.4)#, min_dist=0.2, spread = 1.3)

In [ ]:
'''
https://www.nature.com/articles/s41586-024-07944-6/figures/10
'''
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    subset2, 
    color=['Rag1', 'Rag2', 'Bcl2', #DN4
           'Cd4', 'Cd8a', 'Cd8b1', 'Ccr9', #DP, 
           'Cd52', #DP Q2
           'Cd2', 'Cd28', 'Cd53', # High in DP Sel,
           'Mki67',
           'Bach2', 'Lef1', 'Bcl11b', 
           'Itga4', 
           'Hells', 'Tcf19', 
           'Cd69', 
           'Ikzf2', 'Top2a', 'Tcf7',
           'Mki67', 
          ],   
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False, 
    vmax='p99'
)

In [ ]:
#0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 
#1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0

for resolution_parameter in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0]:
    sc.tl.leiden(subset2, resolution=resolution_parameter, random_state=42, key_added='leiden_'+str(resolution_parameter))

In [ ]:
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4', 'leiden_0.5', 'leiden_0.6', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'

sc.set_figure_params(dpi=80, dpi_save=300, color_map='viridis', vector_friendly=True, transparent=True)
sc.pl.umap(
    subset2, 
    color=[
        'leiden_0.1', 'leiden_0.2', 'leiden_0.3', 'leiden_0.4', 'leiden_0.5',
        'leiden_0.6', 'leiden_0.7', 'leiden_0.8', 'leiden_0.9', 'leiden_1.0',
        'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5',
        'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'
           ], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=4,
    wspace = 0.7,
    outline_width=[0.6, 0.05],
    #size=35,
    frameon=False,
    add_outline=True,
    sort_order = False
)

In [ ]:
#'leiden_1.1', 'leiden_1.2', 'leiden_1.3','leiden_1.4', 'leiden_1.5', 'leiden_1.6', 'leiden_1.7', 'leiden_1.8', 'leiden_1.9', 'leiden_2.0'
#'leiden_0.1', 'leiden_0.2', 'leiden_0.3','leiden_0.4',
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    subset2, 
    color=['leiden_0.1'], 
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False,
    legend_loc='on data'
)

In [ ]:
cell_type_groups = {
    'T:DP(Q)': ['0', '1'],
    #'T:DP(Sel)': [ '4' ],
}

cluster_to_cell_type = {cluster: cell_type for cell_type, clusters in cell_type_groups.items() for cluster in clusters}

In [ ]:
subset2.obs['cell_type_subset'] = subset2.obs['leiden_0.1'].map(cluster_to_cell_type).fillna('DP(P)')

In [ ]:
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    subset2, 
    color=['cell_type_subset', 
          ],
    palette=user_defined_palette,  
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    sort_order = False, 
)

## Map Subsets

In [ ]:
adata.obs['cell_type_subset'].value_counts().sort_index()

In [ ]:
temp = adata[~adata.obs['cell_type_subset'].isin(['sub1', 'sub2'])].obs['cell_type_subset'].copy()
temp

In [ ]:
with_renamed_subsets = pd.concat([subset1.obs['cell_type_subset'], 
                                  subset2.obs['cell_type_subset'],
                                  temp])

In [ ]:
adata.obs['cell_type_subset'] = ''

In [ ]:
adata.obs['cell_type_subset'][adata.obs.index.isin(with_renamed_subsets.index) == True] = with_renamed_subsets

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=1000, n_bins=20, flavor='seurat',  inplace=True)

In [ ]:
sc.tl.pca(adata, n_comps=75, svd_solver='arpack', random_state=42, use_highly_variable=True)

In [ ]:
sce.pp.harmony_integrate(adata, ['sample'], max_iter_harmony = 20)

In [ ]:
sc.pp.neighbors(adata, n_neighbors=30, random_state=42, use_rep='X_pca_harmony')

In [ ]:
sc.tl.umap(adata, random_state = 42) # min_dist=0.2, spread = 1.3,

In [ ]:
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    adata, 
    color=['cell_type_subset'],   
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    size = 10,
    sort_order = False, 
    #vmax='p99'
)

In [ ]:
sc.set_figure_params(dpi=100, dpi_save=300, color_map='viridis', vector_friendly=False, transparent=True)
sc.pl.umap(
    adata, 
    color=['cell_type_subset', 'Foxp3', 'Trdc', 'Tcrg-C4', 'Tcrg-V1', 'Cd2', 'Hdac9'],   
    color_map='Spectral_r', 
    use_raw=False,
    ncols=5,
    wspace = 0.3,
    outline_width=[0.6, 0.05],
    frameon=False,
    add_outline=True,
    size = 10,
    sort_order = False, 
    #vmax='p99'
)

In [ ]:
sc.tl.rank_genes_groups(adata,
groupby='cell_type_subset', 
method='wilcoxon', 
use_raw=False)

result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names']}).head(150) #'scores', 'logfoldchanges', 'pvals_adj'

# df.to_csv(file_outputs + '0_umbrella_gene_rank.csv', index = False)
df.head(50)


In [ ]:
adata

In [ ]:
adata.write_h5ad(h5ad + '4_dn_dp.h5ad')